# EquiCave: train the equivariant multi-task pocket network (Colab / any CUDA machine)

Runtime: GPU (A100 or V100). The notebook clones the repository, installs the dependencies, builds the data
cache and runs `python -m training pockets-net`. Nothing here calls P2Rank, fpocket or Java.

Stages 1-3 are CPU work (about an hour for 1500 structures on a Colab CPU); stage 4 needs the GPU.

In [ ]:
REPO = 'https://github.com/Ruskinkot1/equicave'   # private: use a token or upload the checkout
BRANCH = 'claude/dazzling-fermat-x3au2t'
!git clone --branch $BRANCH $REPO equicave 2>/dev/null || echo 'clone the repository manually if it is private'
%cd equicave
!pip -q install numpy scipy pandas scikit-learn lightgbm pyyaml transformers safetensors
import os; os.environ['PYTHONPATH'] = 'src:.'
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"

In [ ]:
# 1. tests must be green before anything else
!PYTHONPATH=src:. python -m pytest tests -q

In [ ]:
# 2. data: manifest from RCSB (CC0) and the structures (skip if data/processed/manifest.csv is already in the checkout)
!PYTHONPATH=src:. python scripts/data/build_manifest.py --n 3200 --seed 0
!PYTHONPATH=src:. python scripts/data/fetch_structures.py --jobs 8

In [ ]:
# 3. native candidates and the LightGBM ranker (CPU)
!PYTHONPATH=src:. python scripts/train/build_native.py --jobs 4
!PYTHONPATH=src:. python scripts/train/train_ranker.py --seeds 5 --ablate --model models/ranker_native.txt

In [ ]:
# 4. the network: one fold, full config. The first call also builds the feature cache (ESM-2 650M on the GPU).
!PYTHONPATH=src:. python -m training pockets-net --config training/configs/pockets_net.yaml \
    --out runs/training/net --device cuda --set split.val_fold=0 optim.seed=0 tag=full

In [ ]:
# 5. seeds, folds, out-of-fold features for the hybrid ranker, and the ablation grid
!for s in 1 2; do PYTHONPATH=src:. python -m training pockets-net --device cuda --out runs/training/net --set optim.seed=$s split.val_fold=0 tag=full; done
!PYTHONPATH=src:. python -m training pockets-net --device cuda --set mode=oof tag=full
!PYTHONPATH=src:. python scripts/train/train_ranker.py --features-extra net --seeds 5
!bash scripts/train/run_ablations.sh runs/training/ablations 0 3

In [ ]:
# 6. evaluation on every benchmark with one protocol
!PYTHONPATH=src:. python scripts/data/fetch_eval_sets.py
!for s in heldout coach420 holo4k; do PYTHONPATH=src:. python scripts/eval/evaluate.py --set $s \
    --ranker models/ranker_native.txt --net runs/training/net/full_fold0_seed0/model.pt --jobs 4; done
!cat docs/results/eval_heldout.md

## What to report back
`runs/training/*/model_card.json` (config, metrics, geometry constants), `docs/results/*.md`, and the
`history.json` curves. Do not put numbers from a run that did not converge into a table: write "not run".